In [ ]:
from pathlib import Path

import cobra
import pandas as pd

pool_factors = [0.50, 0.75, 1.00, 1.25, 1.50]


def run_protein_pool_sensitivity(model, sources, base_media, calibrated_pool,
    carbon_bound=1000.0):

    output_dict = {}
    output_index = []

    # Get carbon source names
    for source in sources:
        reaction = model.reactions.get_by_id(source)
        carbon_source_name = reaction.name.replace(' exchange', '')
        output_dict[carbon_source_name] = []        

    # Run each protein-pool multiplier
    for factor in pool_factors:
        pool_value = calibrated_pool * factor
        model.reactions.DM_usage.bounds = (-pool_value, 0.0)

        model_id = f"pool_{factor:.2f}x"
        output_index.append(model_id)

        print(
            f"\nRunning {model_id}: "
            f"DM_usage = {-pool_value}"
        )

        for source in sources:
            reaction = model.reactions.get_by_id(source)

            carbon_source_name = reaction.name.replace(' exchange', '')

            media = base_media.copy()
            media.update({source: carbon_bound})

            with model:
                model.medium = media
                solution = model.optimize()

                if solution.status == "optimal":
                    value = solution.objective_value

                else:
                    value = float("nan")

                    print(
                        f"{model_id} failed on "
                        f"{source}: "
                        f"{solution.status}"
                    )

            output_dict[carbon_source_name].append(value)

    df = pd.DataFrame(output_dict, index=output_index).transpose()
    return df

In [ ]:
"""
carved PAO1
"""


base_media = {
    'EX_cd2_e': 1000.0, 'EX_cobalt2_e': 1000.0, 'EX_cu2_e': 1000.0, 'EX_co2_e': 1000.0, 'EX_fe3_e': 1000.0,
    'EX_h_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_k_e': 1000.0, 'EX_mg2_e': 1000.0, 'EX_mn2_e': 1000.0,
    'EX_na1_e': 1000.0, 'EX_nh4_e': 1000.0, 'EX_o2_e': 1000.0, 'EX_pi_e': 1000.0, 'EX_so4_e': 1000.0,
    'EX_zn2_e': 1000.0, 'EX_ca2_e': 1000.0, 'EX_cl_e': 1000.0
}

sources = [
    'EX_4abut_e', 'EX_ac_e', 'EX_cit_e', 'EX_ala__D_e', 'EX_fru_e', 'EX_glc__D_e',
    'EX_mnl_e', 'EX_man_e', 'EX_glcn_e', 'EX_glyc_e', 'EX_ala__L_e', 'EX_asp__L_e',
    'EX_his__L_e', 'EX_lac__L_e', 'EX_leu__L_e', 'EX_orn_e', 'EX_phe__L_e',
    'EX_pro__L_e', 'EX_ser__L_e', 'EX_akg_e', 'EX_ptrc_e', 'EX_succ_e',
    'EX_tre_e', 'EX_asn__L_e', 'EX_cellb_e', 'EX_h2o_e', 'EX_glu__L_e',
    'EX_thr__L_e', 'EX_crn_e'
]


ORIGINAL_EMMAI = Path(
    "Supplementary file 1/PAO1/ecPAO1.xml"
)

OUTPUT_FILE = Path(
    "protein_pool_sensitivity.csv"
)


calibrated_pool = 1.35

output_dict = {}
output_index = []

model = cobra.io.read_sbml_model(ORIGINAL_EMMAI)

for source in sources:

    reaction = model.reactions.get_by_id(source)
    carbon_source_name = reaction.name.replace(' exchange', '')
    output_dict[carbon_source_name] = []

for factor in pool_factors:
    pool_value = calibrated_pool * factor
    model.reactions.DM_usage.bounds = (-pool_value, 0.0)
    model_id = f"pool_{factor:.2f}x"
    output_index.append(model_id)

    print(
        f"\nRunning {model_id}: "
        f"DM_usage = {-pool_value}"
    )

    for source in sources:
        reaction = model.reactions.get_by_id(source)
        carbon_source_name = (reaction.name.replace(' exchange', ''))
        media = base_media.copy()
        media.update({source: 1000.0})

        with model:
            model.medium = media
            solution = model.optimize()
            output_dict[carbon_source_name].append(solution.objective_value)

df = pd.DataFrame(output_dict, index=output_index)
df = df.transpose()
df.to_csv(OUTPUT_FILE)

print(
    f"\nSaved predictions to:\n"
    f"{OUTPUT_FILE}"
)


In [ ]:
"""
iML1515 and iJO1366
"""

star_media = {
    'EX_ca2_e': 1000.0, 'EX_cbl1_e': 0.01, 'EX_cl_e': 1000.0, 'EX_co2_e': 1000.0, 'EX_cobalt2_e': 1000.0,
    'EX_cu2_e': 1000.0, 'EX_fe2_e': 1000.0, 'EX_fe3_e': 1000.0, 'EX_h_e': 1000.0, 'EX_h2o_e': 1000.0,
    'EX_k_e': 1000.0, 'EX_mg2_e': 1000.0, 'EX_mn2_e': 1000.0, 'EX_mobd_e': 1000.0, 'EX_nh4_e': 1000.0,
    'EX_ni2_e': 1000.0, 'EX_o2_e': 1000.0, 'EX_pi_e': 1000.0, 'EX_so4_e': 1000.0, 'EX_zn2_e': 1000.0
}

sources = [
    'ac', 'glyc', 'akg', 'ala__L', 'pyr', 'fru', 'gsn',
    'acgam', 'fum', 'rib__D', 'lac__L', 'glcn', 'sbt__D',
    'gam', 'mal__L', 'succ', 'malt', 'g6p', 'man', 'tre',
    'xyl__D', 'gal', 'glc__D', 'mnl'
]

IJO_ORIGINAL_EMMAI = Path(
    "Supplementary file 1/iJO1366/eciJO1366.xml"
)

IML_ORIGINAL_EMMAI = Path(
    "Supplementary file 1/iML1515/eciML1515.xml"
)

OUTPUT_FILE = Path(
    "protein_pool_sensitivity.csv"
)

ijo_calibrated_pool = 0.68
iml_calibrated_pool = 0.99

ec_ijo_model = cobra.io.read_sbml_model(IJO_ORIGINAL_EMMAI)
ec_iml_model = cobra.io.read_sbml_model(IML_ORIGINAL_EMMAI)

output_dict = {}
output_index = []

# Use iJO1366 to recover readable carbon-source names.
for source in sources:
    carbon_reaction = f'EX_{source}_e'
    reaction = ec_ijo_model.reactions.get_by_id(carbon_reaction)
    carbon_source_name = reaction.name.replace(' exchange', '')
    output_dict[carbon_source_name] = []

def run_model(model, model_id):
    output_index.append(model_id)
    for source in sources:
        carbon_reaction = f'EX_{source}_e'

        reaction = model.reactions.get_by_id(carbon_reaction)

        media = star_media.copy()
        media.update({carbon_reaction: 1000.0})

        with model:
            model.medium = media
            solution = model.optimize()
            if solution.status == "optimal":
                value = solution.objective_value

            else:
                value = float("nan")
                print(
                    f"{model_id} failed on "
                    f"{source}: "
                    f"{solution.status}"
                )

        carbon_source_name = (reaction.name.replace(' exchange', ''))
        output_dict[carbon_source_name].append(value)

for factor in pool_factors:
    pool_value = (ijo_calibrated_pool * factor)
    ec_ijo_model.reactions.DM_usage.bounds = (-pool_value, 0.0)

    model_id = (f"iJO1366_pool_{factor:.2f}x")

    print(
        f"\nRunning {model_id}: "
        f"DM_usage = {-pool_value}"
    )

    run_model(ec_ijo_model, model_id)

for factor in pool_factors:
    pool_value = (iml_calibrated_pool * factor)
    ec_iml_model.reactions.DM_usage.bounds = (-pool_value, 0.0)

    model_id = (f"iML1515_pool_{factor:.2f}x")

    print(
        f"\nRunning {model_id}: "
        f"DM_usage = {-pool_value}"
    )

    run_model(ec_iml_model, model_id)

df = pd.DataFrame(output_dict, index=output_index)
df = df.transpose()
df.to_csv(OUTPUT_FILE)

print(
    f"\nSaved predictions to:\n"
    f"{OUTPUT_FILE}"
)

In [ ]:
"""
iMO1056
"""

ORIGINAL_MODEL = Path(
    "Supplementary file 1/iMO1056/eciMO1056.xml"
)

OUTPUT_FILE = Path(
    "protein_pool_sensitivity.csv"
)

base_media = {
    'EX_cd2_e': 1000.0, 'EX_co2_e': 1000.0, 'EX_cobalt2_e': 1000.0, 'EX_cu2_e': 1000.0, 'EX_fe2_e': 1000.0,
    'EX_fe3_e': 1000.0, 'EX_h_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_k_e': 1000.0, 'EX_mg2_e': 1000.0,
    'EX_mn2_e': 1000.0, 'EX_n2_e': 1000.0, 'EX_na1_e': 1000.0, 'EX_nh4_e': 1000.0, 'EX_o2_e': 1000.0,
    'EX_pi_e': 1000.0, 'EX_so4_e': 1000.0, 'EX_zn2_e': 1000.0
}

sources = [
    'EX_4abut_e', 'EX_ac_e', 'EX_cit_e', 'EX_ala__D_e', 'EX_fru_e', 'EX_glc_e',
    'EX_mnl_e', 'EX_man_e', 'EX_glcn__D_e', 'EX_glyc_e', 'EX_ala__L_e', 'EX_asp__L_e',
    'EX_his__L_e', 'EX_lac__L_e', 'EX_leu__L_e', 'EX_orn__L_e', 'EX_phe__L_e', 'EX_pro__L_e',
    'EX_ser__L_e', 'EX_akg_e', 'EX_ptrc_e', 'EX_succ_e', 'EX_tre_e', 'EX_asn__L_e',
    'EX_maln_e', 'EX_cellb_e', 'EX_h2o_e', 'EX_glu__L_e', 'EX_thr__L_e', 'EX_btd__RR_e',
    'EX_crn_e'
]


original_model = cobra.io.read_sbml_model(
    ORIGINAL_MODEL
)


df_imo = run_protein_pool_sensitivity(
    model=original_model, sources=sources,
    base_media=base_media, calibrated_pool=1.65,
)


df_imo.to_csv(OUTPUT_FILE)
df_imo


In [ ]:
"""
ecBALT
"""

ORIGINAL_MODEL = Path("Supplementary file 1/BALT/ec_BALT_gapfilled.xml")
BIOLOG_FILE = Path("HILSE-31_growth_growth_rates.xlsx")
OUTPUT_FILE = Path("protein_pool_sensitivity.csv")

base_media = {
    'EX_cd2_e': 1000.0, 'EX_cobalt2_e': 1000.0, 'EX_cu2_e': 1000.0, 'EX_fe2_e': 1000.0, 'EX_fe3_e': 1000.0,
    'EX_h_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_k_e': 1000.0, 'EX_mg2_e': 1000.0, 'EX_mn2_e': 1000.0,
    'EX_na1_e': 1000.0, 'EX_nh4_e': 1000.0, 'EX_o2_e': 1000.0, 'EX_pi_e': 1000.0, 'EX_so4_e': 1000.0,
    'EX_zn2_e': 1000.0, 'EX_ca2_e': 1000.0, 'EX_cl_e': 1000.0
}

biolog_31 = pd.read_excel(BIOLOG_FILE, index_col=22)

sources = []
reaction_map = {}

for reaction in biolog_31.index:
    if isinstance(reaction, str):
        reaction_bigg = reaction.replace("(e)", "_e")
        reaction_map[reaction_bigg] = reaction

sources = list(reaction_map.keys())

balt_model = cobra.io.read_sbml_model(ORIGINAL_MODEL)

df_balt = run_protein_pool_sensitivity(
    model=balt_model, sources=sources, base_media=base_media, calibrated_pool=0.105
)

carbon_sources = []
mean_max_rates = []

for source in sources:
    reaction_original = reaction_map[source]
    carbon_sources.append(biolog_31.at[reaction_original, "carbon_source"])
    mean_max_rates.append(biolog_31.at[reaction_original, "mean_max_rate"])

df_balt.insert(0, "carbon_source", carbon_sources)
df_balt.insert(1, "mean_max_rate", mean_max_rates)

df_balt.to_csv(OUTPUT_FILE)
df_balt

In [ ]:
"""
PRET 
"""

ORIGINAL_MODEL = Path("Supplementary file 1/PRET/ec_PRET_gapfilled.xml")
BIOLOG_FILE = Path("HILSE-52_growth_growth_rates.xlsx")
OUTPUT_FILE = Path("protein_pool_sensitivity.csv")

base_media = {
    'EX_cd2_e': 1000.0, 'EX_cobalt2_e': 1000.0, 'EX_cu2_e': 1000.0, 'EX_fe2_e': 1000.0, 'EX_fe3_e': 1000.0,
    'EX_h_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_k_e': 1000.0, 'EX_mg2_e': 1000.0, 'EX_mn2_e': 1000.0,
    'EX_n2_e': 1000.0, 'EX_na1_e': 1000.0, 'EX_nh4_e': 1000.0, 'EX_o2_e': 1000.0, 'EX_pi_e': 1000.0,
    'EX_so4_e': 1000.0, 'EX_zn2_e': 1000.0, 'EX_ca2_e': 1000.0, 'EX_cl_e': 1000.0,
}

biolog_52 = pd.read_excel(BIOLOG_FILE, index_col=22)
reaction_map = {}

for reaction in biolog_52.index:
    if isinstance(reaction, str):
        reaction_bigg = reaction.replace("(e)", "_e")
        reaction_map[reaction_bigg] = reaction

sources = list(reaction_map.keys())
pret_model = cobra.io.read_sbml_model(ORIGINAL_MODEL)

valid_sources = []

for source in sources:
    try:
        pret_model.reactions.get_by_id(source)
        valid_sources.append(source)

    except KeyError:
        print(f"{source} not found in PRET model.")

df_pret = run_protein_pool_sensitivity(
    model=pret_model, sources=valid_sources,
    base_media=base_media, calibrated_pool=0.082,
)

carbon_sources = []
mean_max_rates = []

for source in valid_sources:
    reaction_original = reaction_map[source]
    carbon_sources.append(biolog_52.at[reaction_original, "carbon_source"])
    mean_max_rates.append(biolog_52.at[reaction_original, "mean_max_rate"])

df_pret.insert(0, "carbon_source", carbon_sources)
df_pret.insert(1, "mean_max_rate", mean_max_rates)
df_pret.to_csv(OUTPUT_FILE)

df_pret